# Phase 3: Analysis

Three patterns from Phase 2: Q4 spending surge, amendment growth, and vendor concentration. This notebook tests each with evidence and ends with recommendations.

**Scope**: National Defence excluded. Only valid `reporting_period` rows. Eras analyzed separately where field availability differs.

In [ ]:
import os
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

load_dotenv(override=True)
dataset_path = os.getenv("LOCAL_DATASET_PATH")
con = duckdb.connect(database=":memory:")

con.execute("""
    CREATE TEMP TABLE contracts AS
    SELECT * FROM read_csv_auto(
        '{0}', delim=',', header=true, strict_mode=false, all_varchar=true, parallel=false
    )
""".format(dataset_path))

# Analysis view: excludes Defence, casts types, filters to valid reporting periods
con.execute("""
    CREATE TEMP VIEW analysis AS
    SELECT *,
        TRY_CAST(contract_value AS DOUBLE) AS val,
        TRY_CAST(original_value AS DOUBLE) AS orig_val,
        TRY_CAST(amendment_value AS DOUBLE) AS amend_val,
        LEFT(reporting_period, 9) AS fiscal_year,
        RIGHT(reporting_period, 2) AS quarter,
        CASE 
            WHEN reporting_period < '2019-2020' THEN 'Pre-2019'
            WHEN reporting_period >= '2019-2020' AND reporting_period < '2022-2023' THEN '2019-2022'
            WHEN reporting_period >= '2022-2023' THEN 'Post-2022'
        END AS era
    FROM contracts
    WHERE owner_org_title NOT LIKE 'National Defence%'
      AND reporting_period LIKE '____-____-Q_'
""")

con.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT procurement_id) AS unique_ids FROM analysis").pl()

## The three reporting eras

| Era | What became mandatory |
|---|---|
| Pre-2019 | Only `reference_number`. Many fields 35-97% missing |
| 2019-2022 | Core fields: `procurement_id`, `contract_value`, `commodity_type`, `instrument_type` |
| Post-2022 | Process fields: `solicitation_procedure`, `vendor_postal_code`, `contracting_entity` |

In [ ]:
# Era breakdown
con.sql("""
    SELECT era,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='C' THEN 1 ELSE 0 END) AS contracts,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='C' THEN val ELSE 0 END)/1e9, 2) AS contract_val_B
    FROM analysis
    GROUP BY era ORDER BY era
""").pl()

### Data quality note: vendor name inconsistency

Same vendor appears under different spellings. Our insights do not depend on exact vendor matching, so we proceed as-is.

In [ ]:
# Duplicate vendor names
con.sql("""
    SELECT a.vendor_name AS spelling_1, b.vendor_name AS spelling_2,
           a.cnt AS count_1, b.cnt AS count_2
    FROM (SELECT vendor_name, COUNT(*) AS cnt FROM analysis WHERE vendor_name IS NOT NULL GROUP BY vendor_name HAVING COUNT(*)>50) a
    JOIN (SELECT vendor_name, COUNT(*) AS cnt FROM analysis WHERE vendor_name IS NOT NULL GROUP BY vendor_name HAVING COUNT(*)>50) b
    ON UPPER(REPLACE(REPLACE(a.vendor_name,'.',''),' ','')) = UPPER(REPLACE(REPLACE(b.vendor_name,'.',''),' ',''))
    AND a.vendor_name < b.vendor_name
    ORDER BY a.cnt + b.cnt DESC
    LIMIT 10
""").pl()

---

## Insight 1: Q4 spending surge

Canada's fiscal year ends March 31. If departments rush to spend remaining budget, Q4 (Jan-Mar) should show higher contract counts and values.

In [ ]:
# Quarterly pattern -- contracts only, excluding Defence
q_overall = con.sql("""
    SELECT quarter,
        COUNT(*) AS contracts,
        ROUND(SUM(val)/1e9, 2) AS total_value_B,
        ROUND(AVG(val), 0) AS avg_value
    FROM analysis
    WHERE instrument_type = 'C'
    GROUP BY quarter ORDER BY quarter
""").pl()
q_overall

In [64]:
# Q4 surge - count and average value
fig = make_subplots(rows=1, cols=2, subplot_titles=("Contract Count", "Average Contract Value"))
colors = ['#4878A8', '#4878A8', '#4878A8', '#D04040']

fig.add_trace(go.Bar(x=q_overall["quarter"].to_list(), y=q_overall["contracts"].to_list(),
    marker_color=colors, text=[f'{v/1000:.0f}K' for v in q_overall["contracts"].to_list()], textposition='outside', cliponaxis=False), row=1, col=1)

fig.add_trace(go.Bar(x=q_overall["quarter"].to_list(), y=q_overall["avg_value"].to_list(),
    marker_color=colors, text=[f'${v/1000:.0f}K' for v in q_overall["avg_value"].to_list()], textposition='outside', cliponaxis=False), row=1, col=2)

fig.update_layout(title_text="Q4 (Jan-Mar) has more contracts and higher average values", showlegend=False, height=600)
fig.show()

32% more new contracts in Q4 than the Q1-Q3 average (contracts only). Across all transaction types (including amendments), the surge is +38%. The volume pattern is clear.

In [65]:
# Q4 pattern by era
q_era = con.sql("""
    SELECT era, quarter, COUNT(*) AS contracts, ROUND(AVG(val), 0) AS avg_value
    FROM analysis WHERE instrument_type = 'C' AND era IS NOT NULL
    GROUP BY era, quarter ORDER BY era, quarter
""").pl()

fig = make_subplots(rows=1, cols=3, subplot_titles=['Pre-2019', '2019-2022', 'Post-2022'], shared_yaxes=True)
for i, era in enumerate(['Pre-2019', '2019-2022', 'Post-2022']):
    d = q_era.filter(pl.col("era") == era)
    colors = ['#4878A8', '#4878A8', '#4878A8', '#D04040']
    fig.add_trace(go.Bar(x=d["quarter"].to_list(), y=d["avg_value"].to_list(),
        marker_color=colors, text=[f'${v/1000:.0f}K' for v in d["avg_value"].to_list()],
        textposition='outside', cliponaxis=False, showlegend=False), row=1, col=i+1)

fig.update_layout(title_text="Q4 spending surge by era - the pattern intensified post-2022", height=500)
fig.update_yaxes(title_text="Avg Contract Value ($)", row=1, col=1)
fig.show()

In [ ]:
# Q4 share of contracts and value by era
con.sql("""
    SELECT 
        COALESCE(era, 'Overall') AS period,
        COUNT(*) AS total_contracts,
        SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END) AS q4_contracts,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct_count,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN val ELSE 0 END)*100.0/SUM(val), 1) AS q4_pct_value,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END) / NULLIF(AVG(CASE WHEN quarter!='Q4' THEN val END), 0), 2) AS q4_multiplier
    FROM analysis
    WHERE instrument_type = 'C'
    GROUP BY ROLLUP(era)
    ORDER BY CASE WHEN era IS NULL THEN 'ZZZ' ELSE era END
""").pl()

Q4 accounts for 30.6% of contracts and 34.7% of value (expected: 25%). The pattern shifted across eras: pre-2019 was about volume (35.4% of contracts in Q4, but 0.91x multiplier on value). Post-2022 flipped -- only 23.8% of contracts in Q4, but 1.89x higher average value.

In [ ]:
# Top 10 departments by Q4 value multiplier (post-2019)
con.sql("""
    SELECT 
        SPLIT_PART(owner_org_title, ' | ', 1) AS department,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END) /
              NULLIF(AVG(CASE WHEN quarter!='Q4' THEN val END), 0), 1) AS q4_multiplier,
        COUNT(*) AS total_contracts
    FROM analysis
    WHERE instrument_type = 'C' AND reporting_period >= '2019-2020'
    GROUP BY owner_org_title
    HAVING COUNT(*) >= 500
    ORDER BY q4_multiplier DESC
    LIMIT 10
""").pl()

In [68]:
# Department Q4 multipliers
dept_q4 = con.sql("""
    SELECT SPLIT_PART(owner_org_title, ' | ', 1) AS department,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END) / NULLIF(AVG(CASE WHEN quarter!='Q4' THEN val END), 0), 1) AS q4_multiplier
    FROM analysis WHERE instrument_type = 'C' AND reporting_period >= '2019-2020'
    GROUP BY owner_org_title HAVING COUNT(*) >= 500
    ORDER BY q4_multiplier DESC LIMIT 10
""").pl()

fig = px.bar(dept_q4.to_pandas(), y="department", x="q4_multiplier", orientation='h',
    color="q4_multiplier", color_continuous_scale=["#4878A8", "#E8A040", "#D04040"],
    text="q4_multiplier", title="Which departments spend the most disproportionately in Q4?")
fig.add_vline(x=1.0, line_dash="dash", line_color="gray", annotation_text="No difference")
fig.update_traces(texttemplate='%{text}x', textposition='outside')
fig.update_layout(yaxis=dict(autorange="reversed"), height=450, xaxis_title="Q4 avg / Other quarters avg",
    yaxis_title="", coloraxis_showscale=False)
fig.show()

In [ ]:
# Non-competitive rate: Q4 vs other quarters (post-2019)
con.sql("""
    SELECT 
        CASE WHEN quarter = 'Q4' THEN 'Q4 (Jan-Mar)' ELSE 'Q1-Q3 (Apr-Dec)' END AS period,
        COUNT(*) AS total_contracts,
        SUM(CASE WHEN solicitation_procedure='TN' THEN 1 ELSE 0 END) AS non_competitive,
        ROUND(SUM(CASE WHEN solicitation_procedure='TN' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS non_comp_pct
    FROM analysis
    WHERE instrument_type = 'C' AND reporting_period >= '2019-2020'
    GROUP BY period ORDER BY period
""").pl()

In [ ]:
# Q4 vs other quarters by commodity type (post-2019)
con.sql("""
    SELECT 
        commodity_type,
        COUNT(*) AS total,
        SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END) AS q4_count,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg_val,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg_val,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END) / 
              NULLIF(AVG(CASE WHEN quarter!='Q4' THEN val END), 0), 2) AS q4_multiplier
    FROM analysis
    WHERE instrument_type = 'C' AND reporting_period >= '2019-2020'
      AND commodity_type IS NOT NULL
    GROUP BY commodity_type
    ORDER BY q4_multiplier DESC
""").pl()

In [71]:
# Q4 value surge by commodity type - grouped bar
comm_q4 = con.sql("""
    SELECT CASE commodity_type WHEN 'S' THEN 'Services' WHEN 'G' THEN 'Goods' WHEN 'C' THEN 'Construction' END AS commodity,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg
    FROM analysis WHERE instrument_type = 'C' AND reporting_period >= '2019-2020' AND commodity_type IS NOT NULL
    GROUP BY commodity_type ORDER BY commodity_type
""").pl().to_pandas()

fig = go.Figure()
fig.add_trace(go.Bar(name='Q1-Q3 avg', x=comm_q4['commodity'], y=comm_q4['other_avg'], marker_color='#4878A8',
    text=[f'${v/1000:.0f}K' for v in comm_q4['other_avg']], textposition='outside', cliponaxis=False))
fig.add_trace(go.Bar(name='Q4 avg', x=comm_q4['commodity'], y=comm_q4['q4_avg'], marker_color='#D04040',
    text=[f'${v/1000:.0f}K' for v in comm_q4['q4_avg']], textposition='outside', cliponaxis=False))
fig.update_layout(barmode='group', title="Q4 value surge by commodity type - construction is 5.8x higher",
    yaxis_title="Average Contract Value ($)", height=600)
fig.show()

In [ ]:
# Q4 rush by transaction type: new contracts vs amendments vs SOSAs (post-2019)
con.sql("""
    SELECT 
        instrument_type,
        COUNT(*) AS total,
        SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END) AS q4_count,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN val ELSE 0 END)/1e9, 2) AS q4_value_B,
        ROUND(SUM(CASE WHEN quarter!='Q4' THEN val ELSE 0 END)/1e9, 2) AS other_value_B
    FROM analysis
    WHERE reporting_period >= '2019-2020' AND instrument_type IS NOT NULL
    GROUP BY instrument_type
    ORDER BY total DESC
""").pl()

In [73]:
# Q4 concentration by instrument type
inst_q4 = con.sql("""
    SELECT CASE instrument_type WHEN 'C' THEN 'New Contracts' WHEN 'A' THEN 'Amendments' WHEN 'SOSA' THEN 'SOSAs' END AS type,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct
    FROM analysis WHERE reporting_period >= '2019-2020' AND instrument_type IS NOT NULL
    GROUP BY instrument_type ORDER BY q4_pct DESC
""").pl().to_pandas()

fig = px.bar(inst_q4, x='type', y='q4_pct', text='q4_pct', color='q4_pct',
    color_continuous_scale=["#4878A8", "#D04040"], title="Q4 share by transaction type")
fig.add_hline(y=25, line_dash="dash", line_color="gray", annotation_text="Expected if even (25%)")
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=500, yaxis_title="% of transactions in Q4", xaxis_title="", coloraxis_showscale=False)
fig.show()

In [ ]:
# Q4 surge by commodity type AND instrument type (post-2019)
con.sql("""
    SELECT 
        commodity_type,
        instrument_type,
        COUNT(*) AS total,
        SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END) AS q4_count,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct
    FROM analysis
    WHERE reporting_period >= '2019-2020' 
      AND commodity_type IS NOT NULL 
      AND instrument_type IN ('C', 'A')
    GROUP BY commodity_type, instrument_type
    ORDER BY commodity_type, instrument_type
""").pl()

The Q4 rush operates through two channels: new contracts and amendments. Both are more concentrated in Q4. Any oversight response needs to address both.

In [75]:
# Year-over-year Q4 trend
q4_trend = con.sql("""
    SELECT fiscal_year,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg_val,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg_val
    FROM analysis
    WHERE instrument_type = 'C' AND reporting_period >= '2019-2020' AND fiscal_year <= '2024-2025'
    GROUP BY fiscal_year ORDER BY fiscal_year
""").pl().to_pandas()

fig = go.Figure()
fig.add_trace(go.Scatter(x=q4_trend['fiscal_year'], y=q4_trend['other_avg_val'], mode='lines+markers',
    name='Q1-Q3 avg value', line=dict(color='#4878A8', width=2)))
fig.add_trace(go.Scatter(x=q4_trend['fiscal_year'], y=q4_trend['q4_avg_val'], mode='lines+markers',
    name='Q4 avg value', line=dict(color='#D04040', width=3)))
fig.update_layout(title="Q4 vs Q1-Q3 average contract value by fiscal year", yaxis_title="Average Contract Value ($)",
    xaxis_title="Fiscal Year", height=400, hovermode='x unified')
fig.show()

### Insight 1 summary

*Volume measured across all transaction types. Value measured on new contracts only (amendment values are cumulative totals, not new spending).*

**Volume**: Q4 sees 38% more procurement activity than the Q1-Q3 average (all transaction types). New contracts alone show +32%.

**Value**: 34.8% of total contract value lands in Q4 (expected: 25%). Q4 average is $215K vs $177K. Post-2022, the multiplier is 1.89x. Construction hit hardest at 5.84x.

**Two channels**: New contracts (27.5% in Q4) and amendments (32.7% in Q4) both drive the surge. Sole-source rate is higher in Q4 (41.3% vs 39.1%).

**Recommendation**: Flag high-value Q4 construction contracts. Track Q4 sole-source and amendment rates by department. Consider multi-year budgeting.

**Caveat**: `reporting_period` is when the contract was *reported*, not *awarded*. Only mandatory after 2019-01-01.

---

## Insight 2: Amendment growth

Amendment rate jumped from 16% to 25% since 2019, and some contracts grow far beyond their original scope.

In [ ]:
# Amendment rate by era
con.sql("""
    SELECT 
        COALESCE(era, 'Overall') AS period,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='C' THEN 1 ELSE 0 END) AS contracts,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amendment_rate_pct
    FROM analysis
    GROUP BY ROLLUP(era)
    ORDER BY CASE WHEN era IS NULL THEN 'ZZZ' ELSE era END
""").pl()

In [77]:
# Amendment rate by fiscal year
amend_by_year = con.sql("""
    SELECT fiscal_year, ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_pct
    FROM analysis WHERE fiscal_year >= '2017-2018' AND fiscal_year <= '2024-2025' AND fiscal_year != '2018-2018'
    GROUP BY fiscal_year ORDER BY fiscal_year
""").pl().to_pandas()

fig = px.bar(amend_by_year, x='fiscal_year', y='amend_pct', text='amend_pct',
    title="Amendment rate by fiscal year (excl. Defence)", color_discrete_sequence=['#4878A8'])
fig.add_hline(y=20, line_dash="dash", line_color="gray", annotation_text="20% baseline")
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(xaxis_title="Fiscal Year", yaxis_title="Amendment rate (%)", height=500)
fig.show()

Overall amendment rate is ~21%, masking a clear shift: 16.2% pre-2019, then ~25% and stable. The rate alone does not reveal whether these are small extensions or major scope changes.

In [ ]:
# Distribution of contract growth via amendments
growth = con.sql("""
    SELECT 
        CASE 
            WHEN growth_pct < 0 THEN 'Decreased'
            WHEN growth_pct = 0 THEN 'No change'
            WHEN growth_pct <= 25 THEN '1-25%'
            WHEN growth_pct <= 50 THEN '26-50%'
            WHEN growth_pct <= 100 THEN '51-100%'
            WHEN growth_pct <= 500 THEN '101-500%'
            ELSE '500%+'
        END AS growth_bucket,
        COUNT(*) AS contracts,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1) AS pct
    FROM (
        SELECT procurement_id,
            ROUND((TRY_CAST(MAX(contract_value) AS DOUBLE) / 
                   NULLIF(TRY_CAST(MIN(original_value) AS DOUBLE), 0) - 1) * 100, 0) AS growth_pct
        FROM analysis
        WHERE procurement_id IS NOT NULL
        GROUP BY procurement_id
        HAVING COUNT(*) > 1 AND COUNT(DISTINCT instrument_type) > 1
    ) sub
    WHERE growth_pct IS NOT NULL
    GROUP BY growth_bucket
    ORDER BY MIN(growth_pct)
""").pl()
growth

In [79]:
# Amendment growth distribution
color_map = {'Decreased': '#6BAF6B', 'No change': '#888888', '1-25%': '#4878A8', '26-50%': '#4878A8',
             '51-100%': '#4878A8', '101-500%': '#E8A040', '500%+': '#D04040'}
growth_pd = growth.to_pandas()
growth_pd['color'] = growth_pd['growth_bucket'].map(color_map)

fig = px.bar(growth_pd, y='growth_bucket', x='pct', orientation='h', text='pct',
    title="How much do amendments grow contract values?", color='growth_bucket',
    color_discrete_map=color_map)
fig.update_traces(texttemplate='%{text}% (%{customdata[0]:,})', textposition='outside',
    customdata=growth_pd[['contracts']].values)
fig.update_layout(yaxis=dict(autorange="reversed", title=""), xaxis_title="% of amended contracts",
    height=400, showlegend=False)
fig.show()

~24% of amended contracts more than double in value. Checking the most extreme cases and which commodity types are hit hardest.

In [ ]:
# Top 10 contracts by absolute growth (original > $100K)
con.sql("""
    SELECT 
        procurement_id,
        vendor_name,
        SPLIT_PART(owner_org_title, ' | ', 1) AS department,
        amendment_count,
        ROUND(original_M, 1) AS original_M,
        ROUND(final_M, 1) AS final_M,
        ROUND(growth_pct, 0) AS growth_pct
    FROM (
        SELECT procurement_id,
            MAX(vendor_name) AS vendor_name,
            MAX(owner_org_title) AS owner_org_title,
            COUNT(*) - 1 AS amendment_count,
            TRY_CAST(MIN(original_value) AS DOUBLE)/1e6 AS original_M,
            TRY_CAST(MAX(contract_value) AS DOUBLE)/1e6 AS final_M,
            (TRY_CAST(MAX(contract_value) AS DOUBLE) / 
             NULLIF(TRY_CAST(MIN(original_value) AS DOUBLE), 0) - 1) * 100 AS growth_pct
        FROM analysis
        WHERE procurement_id IS NOT NULL
        GROUP BY procurement_id
        HAVING COUNT(*) > 1
    ) sub
    WHERE original_M > 0.1 AND growth_pct > 100
    ORDER BY final_M - original_M DESC
    LIMIT 10
""").pl()

In [ ]:
# Amendment rate by commodity type (post-2019)
amend_commodity = con.sql("""
    SELECT commodity_type,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_rate_pct
    FROM analysis
    WHERE reporting_period >= '2019-2020' AND commodity_type IS NOT NULL
    GROUP BY commodity_type ORDER BY amend_rate_pct DESC
""").pl()
amend_commodity

In [82]:
# Amendment rate by commodity type
fig = px.bar(amend_commodity.to_pandas(), x='commodity_type', y='amend_rate_pct', text='amend_rate_pct',
    title="Services and construction are amended 3x more than goods",
    color='commodity_type', color_discrete_map={'S': '#D04040', 'C': '#E8A040', 'G': '#4878A8'},
    labels={'commodity_type': 'Commodity Type', 'amend_rate_pct': 'Amendment Rate (%)'})
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=500, showlegend=False)
fig.show()

In [ ]:
# Amendment rate: Q4 vs other quarters (post-2019)
con.sql("""
    SELECT 
        CASE WHEN quarter = 'Q4' THEN 'Q4 (Jan-Mar)' ELSE 'Q1-Q3 (Apr-Dec)' END AS period,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_rate_pct
    FROM analysis
    WHERE reporting_period >= '2019-2020'
    GROUP BY period ORDER BY period
""").pl()

In [84]:
# Q4 vs other amendment rate
fig = go.Figure(go.Bar(x=['Q1-Q3', 'Q4'], y=[23.3, 28.2], marker_color=['#4878A8', '#D04040'],
    text=['23.3%', '28.2%'], textposition='outside', cliponaxis=False))
fig.add_hline(y=25, line_dash="dash", line_color="gray", annotation_text="25%")
fig.update_layout(title="Amendments spike in Q4", yaxis_title="Amendment rate (%)", height=350)
fig.show()

### Insight 2 summary

**Pattern**: 1 in 4 rows is an amendment (up from 1 in 6 pre-2019). Services at 31.8%, construction at 30.4%, goods at 9%. Amendments spike in Q4 (28.2% vs 23.3%).

**Evidence**: Bell Mobility grew from $18.6M to $985.6M (16 amendments). Parsons Inc grew from $49.8M to $991.4M (23 amendments). These effectively bypassed the competitive process.

**Recommendation**: Set amendment thresholds -- if cumulative amendments exceed 50% of original value, require justification or re-compete. Track amendment rates by commodity type. Watch Q4 amendments specifically.

**Caveat**: Growth estimated by comparing `MAX(contract_value)` to `MIN(original_value)` across rows sharing a `procurement_id`.

---

## Insight 3: Vendor concentration amplifies amendment risk

Top 50 vendors hold over half of all spend, and their amendment rate is nearly double the rest.

In [ ]:
# Vendor concentration -- top 10/50/100 share of total spend
conc = con.sql("""
    WITH v AS (
        SELECT vendor_name, SUM(val) as total_val,
               ROW_NUMBER() OVER (ORDER BY SUM(val) DESC) as rn
        FROM analysis WHERE vendor_name IS NOT NULL
        GROUP BY vendor_name
    ),
    gt AS (SELECT SUM(total_val) as g, COUNT(*) as total_vendors FROM v)
    SELECT 
        (SELECT total_vendors FROM gt) as total_vendors,
        (SELECT ROUND(g/1e9, 1) FROM gt) as total_B,
        ROUND((SELECT SUM(total_val) FROM v WHERE rn <= 10) * 100.0 / (SELECT g FROM gt), 1) as top10_pct,
        ROUND((SELECT SUM(total_val) FROM v WHERE rn <= 50) * 100.0 / (SELECT g FROM gt), 1) as top50_pct,
        ROUND((SELECT SUM(total_val) FROM v WHERE rn <= 100) * 100.0 / (SELECT g FROM gt), 1) as top100_pct
""").pl()
conc

In [86]:
# Do top vendors get amended more?
tier_amend = con.sql("""
    WITH v AS (
        SELECT vendor_name, SUM(val) as total_val,
               ROW_NUMBER() OVER (ORDER BY SUM(val) DESC) as rn
        FROM analysis WHERE vendor_name IS NOT NULL GROUP BY vendor_name
    )
    SELECT 
        CASE WHEN v.rn <= 50 THEN 'Top 50 vendors' ELSE 'All others' END as tier,
        COUNT(*) as rows,
        ROUND(100.0 * SUM(CASE WHEN a.instrument_type='A' THEN 1 ELSE 0 END) / COUNT(*), 1) as amend_rate
    FROM analysis a
    JOIN v ON a.vendor_name = v.vendor_name
    GROUP BY CASE WHEN v.rn <= 50 THEN 'Top 50 vendors' ELSE 'All others' END
    ORDER BY MIN(v.rn)
""").pl()

fig = go.Figure(go.Bar(
    x=tier_amend["tier"].to_list(), y=tier_amend["amend_rate"].to_list(),
    marker_color=['#D04040', '#4878A8'],
    text=[f"{v}%" for v in tier_amend["amend_rate"].to_list()],
    textposition="outside", cliponaxis=False))
fig.update_layout(title="Top vendors have nearly double the amendment rate",
    yaxis_title="Amendment rate (%)", template="plotly_white", height=400, margin=dict(t=60))
fig.show()

In [ ]:
# Department dependency -- which rely most on a single vendor?
con.sql("""
    WITH dept_vendor AS (
        SELECT owner_org_title, vendor_name, SUM(val) as vendor_total,
               SUM(SUM(val)) OVER (PARTITION BY owner_org_title) as dept_total,
               ROW_NUMBER() OVER (PARTITION BY owner_org_title ORDER BY SUM(val) DESC) as rn
        FROM analysis WHERE vendor_name IS NOT NULL AND val > 0
        GROUP BY owner_org_title, vendor_name
    )
    SELECT SPLIT_PART(owner_org_title, ' | ', 1) as department,
           vendor_name as top_vendor,
           ROUND(dept_total/1e9, 2) as dept_total_B,
           ROUND(vendor_total/1e9, 2) as vendor_total_B,
           ROUND(100.0 * vendor_total / dept_total, 1) as dependency_pct
    FROM dept_vendor
    WHERE rn = 1 AND dept_total > 1e9
    ORDER BY dependency_pct DESC
    LIMIT 10
""").pl()

### Insight 3 summary

**Pattern**: Top 50 vendors (out of 126,000+) hold ~55% of all spending. Their amendment rate is ~38% -- nearly double the ~21% for others. 18 of 97 departments have >30% of spend with a single vendor.

**Why it matters**: Vendor concentration creates dependency risk and reduces negotiating leverage. The highest-spend vendors are also the ones whose contracts change the most after award, connecting directly to Insight 2.

**Recommendation**: Map vendor dependency for critical services. Diversify the vendor base for recurring categories. Monitor whether top-vendor contracts grow disproportionately through amendments.

---

## The cycle

These three insights form a self-reinforcing loop: year-end budget pressure creates rushed awards (Insight 1). Those contracts grow through amendments (Insight 2), disproportionately benefiting established vendors (Insight 3), who then receive even more spend. The competitive process used for the original award becomes less meaningful over time.

## Federal benchmarks

| Metric | Value | Scope |
|---|---|---|
| Q4 volume surge | +38% vs Q1-Q3 avg | All transaction types |
| Q4 share of contract value | 34.8% (expected: 25%) | Contracts only |
| Q4 avg contract value vs Q1-Q3 | $215K vs $177K | Contracts only |
| Q4 construction multiplier | 5.84x | Contracts only, post-2019 |
| Q4 sole-source rate vs Q1-Q3 | 41.3% vs 39.1% | Contracts only, post-2019 |
| Q4 share of new contracts | 27.5% | Post-2019 |
| Q4 share of amendments | 32.7% | Post-2019 |
| Amendment rate | ~25% | Post-2019 |
| Services amendment rate | 31.4% | Post-2019 |
| Top 50 vendor share of spend | ~55% | All eras |
| Top 50 vendor amendment rate | ~38% (vs ~21% others) | All eras |
| Depts with >30% single-vendor dependency | 18 of 97 | All eras |

## Next steps

1. Do contracts *awarded* in Q4 get amended more than Q1-Q3 contracts later on?
2. Which contract descriptions drive the Q4 volume surge (temporary help, consulting)?
3. Vendor name normalization - fuzzy matching would reveal true concentration.
4. Do top vendors bid low and grow through amendments?